# Reddit API - Restaurant Recommendation Scraper
**belly-button project | INFO 290 | Tubal Theodros**

## Architecture Overview
This notebook builds a **comment-first master database** of SF restaurant recommendations.

Each row in the database is an **individual comment** — not a post — with its own recency score.
This means the LLM receives a flat, ranked list of direct restaurant recommendations
rather than having to dig through nested post structures.

**Pipeline:**
- Daily refresh: scrape posts → LLM classifier filters → flatten to individual comments → store with recency scores
- Per query: search comment database instantly → pass ranked comments to LLM as context
- Ranking: purely by comment recency (1.0 = today, 0.0 = 90 days ago)

https://drive.google.com/drive/folders/1ciXnuzZDcVpAVJVg-72nwWQVDm5_6xhm?usp=drive_link

## 1. Install & Import Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install praw pandas python-dotenv anthropic

In [ ]:
# ============================================================
# Setup — Mount Google Drive and set file paths
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# File paths (all JSONs live in belly button folder on Google Drive)
DRIVE_FOLDER = '/content/drive/MyDrive/belly button'

DB_PATH_V2 = f'{DRIVE_FOLDER}/master_database_v2.json'
EVAL_SNAPSHOT_PATH = f'{DRIVE_FOLDER}/eval_snapshot.json'
QA_VAL_PATH = f'{DRIVE_FOLDER}/qa_validation.json'
QA_TEST_PATH = f'{DRIVE_FOLDER}/qa_test.json'
EVAL_RESULTS_PATH = f'{DRIVE_FOLDER}/eval_results_validation.json'
DB_PATH = f'{DRIVE_FOLDER}/master_database.json'

# V2 corpus metadata
SUBREDDITS_V2 = ['AskSF', 'SFFood']
MAX_AGE_DAYS_V2 = 365
TOP_COMMENTS_V2 = 25

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import praw
import pandas as pd
import os
import json
from datetime import datetime, timezone
from dotenv import load_dotenv
import anthropic

## 2. Credentials

In [ ]:
from google.colab import userdata
reddit = praw.Reddit(
    client_id=userdata.get('REDDIT_CLIENT_ID'),
    client_secret=userdata.get('REDDIT_CLIENT_SECRET'),
    username=userdata.get('REDDIT_USERNAME'),
    password=userdata.get('REDDIT_PASSWORD'),
    user_agent='restaurant_recommender by u/Effective-Street7281'
)
claude = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

## 3. Configuration

In [ ]:
# Best subreddits for SF restaurant recommendations
SUBREDDITS = ['AskSF', 'SFFood']

# Posts to pull per subreddit per feed
POST_LIMIT = 250

# Max age in days for posts AND comments
MAX_AGE_DAYS = 90

# Comments to extract per post
TOP_COMMENTS = 10

# Master database file
DB_PATH = 'master_database.json'

# Words to skip when matching queries against database
SKIP_WORDS = [
    'best', 'good', 'great', 'san', 'francisco', 'sf',
    'restaurant', 'place', 'spot', 'where', 'what', 'any'
]

# Food signals for keyword pre-filter
FOOD_SIGNALS = [
    'restaurant', 'restaurants', 'ramen', 'sushi', 'italian', 'mexican',
    'chinese', 'thai', 'indian', 'pizza', 'burger', 'burgers', 'taco',
    'tacos', 'pho', 'dim sum', 'brunch', 'dining', 'where to eat',
    'best place to eat', 'food rec', 'food recommendation',
    'restaurant recommendation', 'good spot', 'hole in the wall',
    'happy hour', 'cafe', 'eatery', 'cuisine', 'eats', 'izakaya',
    'omakase', 'korean bbq', 'hot pot', 'boba', 'dumplings', 'banh mi',
    'best food', 'good food', 'great food', 'lunch spot', 'dinner spot',
    'breakfast spot', 'foodie', 'michelin', 'speakeasy', 'gastropub',
    'wine bar', 'cocktail bar', 'rooftop bar', 'best bar', 'good bar',
    'steakhouse', 'seafood', 'vegetarian', 'vegan', 'halal', 'kosher',
    'food spots', 'eating', 'must eat', 'must try', 'underrated',
    'hidden gem', 'new restaurant', 'just opened', 'recently opened'
]

## 4. Helper Functions

In [ ]:
def get_age_days(utc_timestamp):
    """Return how many days ago a timestamp was."""
    post_time = datetime.fromtimestamp(utc_timestamp, tz=timezone.utc)
    now = datetime.now(tz=timezone.utc)
    return (now - post_time).days


def format_timestamp(utc_timestamp):
    """Convert UTC timestamp to readable date string."""
    return datetime.fromtimestamp(utc_timestamp, tz=timezone.utc).strftime('%Y-%m-%d')


def compute_recency_score(age_days, max_age=MAX_AGE_DAYS):
    """
    Recency score 0.0 to 1.0.
    1.0 = today, 0.0 = 90 days ago. Linear decay.
    This is applied to COMMENT date, not post date.
    """
    if age_days <= 0:
        return 1.0
    if age_days >= max_age:
        return 0.0
    return round(1.0 - (age_days / max_age), 3)


def keyword_filter(post_title):
    """Fast first-pass: title must contain a food signal."""
    return any(signal in post_title.lower() for signal in FOOD_SIGNALS)


def llm_classifier(post_title, post_body, comments_preview):
    """
    LLM classifier using Claude Haiku.
    Runs ONCE per new post during daily refresh.
    Confirms post is a real restaurant recommendation thread.
    """
    prompt = f"""Is this Reddit post a restaurant recommendation thread where people are asking for or giving specific restaurant suggestions?

Title: {post_title}
Body: {post_body[:200]}
Top comments preview: {comments_preview[:300]}

Answer yes or no only."""

    try:
        response = claude.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=5,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response.content[0].text.strip().lower() == 'yes'
    except Exception as e:
        print(f'LLM classifier error: {e}')
        return False


def extract_comments_as_rows(submission, post_title, post_url, subreddit):
    """
    Extract comments as individual flat rows.
    Each comment becomes one entry in the database.
    Recency score is based on COMMENT date, not post date.
    """
    submission.comments.replace_more(limit=0)
    rows = []

    for comment in submission.comments[:TOP_COMMENTS]:
        age_days = get_age_days(comment.created_utc)

        # Skip comments older than 90 days
        if age_days > MAX_AGE_DAYS:
            continue

        # Skip empty or removed comments
        if not comment.body or comment.body in ['[removed]', '[deleted]']:
            continue

        rows.append({
            'comment_id': comment.id,
            'comment_body': comment.body,
            'comment_date': format_timestamp(comment.created_utc),
            'comment_age_days': age_days,
            'recency_score': compute_recency_score(age_days),
            'comment_upvotes': comment.score,
            'post_title': post_title,
            'post_url': post_url,
            'subreddit': subreddit
        })

    return rows


print('Helper functions loaded.')

Helper functions loaded.


## 5. Build / Refresh Master Database
Each entry in the database is one comment with its own recency score.
Run once to build, then daily to refresh.

In [ ]:
def update_database(db_path=DB_PATH):
    """
    Daily refresh.
    1. Load existing database
    2. Scrape new posts
    3. Keyword filter → LLM classifier (new posts only)
    4. Flatten approved posts into individual comment rows
    5. Remove comments older than 90 days
    6. Save updated database
    """
    # Load existing database
    try:
        with open(db_path, 'r') as f:
            existing = json.load(f)
        existing_comments = {c['comment_id']: c for c in existing['comments']}
        seen_post_ids = set(existing.get('processed_post_ids', []))
        print(f'Loaded existing database: {len(existing_comments)} comments')
    except FileNotFoundError:
        existing_comments = {}
        seen_post_ids = set()
        print('No existing database — building fresh')

    new_comments = 0
    rejected_keyword = 0
    rejected_llm = 0
    new_posts_processed = 0

    for sub_name in SUBREDDITS:
        print(f'\nScraping r/{sub_name}...')
        subreddit = reddit.subreddit(sub_name)

        feeds = [
            subreddit.new(limit=POST_LIMIT),
            subreddit.hot(limit=POST_LIMIT),
            subreddit.top(limit=POST_LIMIT, time_filter='month')
        ]

        seen_in_this_run = set()

        for feed in feeds:
            for post in feed:
                # Skip if already processed
                if post.id in seen_post_ids or post.id in seen_in_this_run:
                    continue
                seen_in_this_run.add(post.id)

                # Skip old posts
                if get_age_days(post.created_utc) > MAX_AGE_DAYS:
                    continue

                # Stage 1: keyword filter
                if not keyword_filter(post.title):
                    rejected_keyword += 1
                    continue

                # Get comments preview for LLM classifier
                post.comments.replace_more(limit=0)
                comments_preview = ' '.join([
                    c.body[:100] for c in post.comments[:3]
                    if c.body not in ['[removed]', '[deleted]']
                ])

                # Stage 2: LLM classifier
                if not llm_classifier(post.title, post.selftext, comments_preview):
                    rejected_llm += 1
                    seen_post_ids.add(post.id)
                    continue

                # Extract comments as individual rows
                comment_rows = extract_comments_as_rows(
                    post, post.title,
                    f'https://reddit.com{post.permalink}',
                    sub_name
                )

                # Add new comments to database
                for row in comment_rows:
                    if row['comment_id'] not in existing_comments:
                        existing_comments[row['comment_id']] = row
                        new_comments += 1

                seen_post_ids.add(post.id)
                new_posts_processed += 1

    # Remove comments older than 90 days
    before = len(existing_comments)
    existing_comments = {
        cid: c for cid, c in existing_comments.items()
        if c['comment_age_days'] <= MAX_AGE_DAYS
    }
    removed = before - len(existing_comments)

    # Save database
    output = {
        'last_updated': datetime.now(tz=timezone.utc).isoformat(),
        'total_comments': len(existing_comments),
        'max_age_days': MAX_AGE_DAYS,
        'processed_post_ids': list(seen_post_ids),
        'comments': list(existing_comments.values())
    }

    with open(db_path, 'w') as f:
        json.dump(output, f, indent=2)

    print(f'\n=== Database Update Complete ===')
    print(f'New posts processed: {new_posts_processed}')
    print(f'New comments added: {new_comments}')
    print(f'Rejected by keyword filter: {rejected_keyword}')
    print(f'Rejected by LLM classifier: {rejected_llm}')
    print(f'Comments removed (>90 days): {removed}')
    print(f'Total comments in database: {len(existing_comments)}')

    return list(existing_comments.values())


# Run to build or refresh the database
comments = update_database()
print(f'\nDatabase ready with {len(comments)} comment rows')

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



No existing database — building fresh

Scraping r/AskSF...


ResponseException: received 401 HTTP response

## 6. View Database as DataFrame
Each row is one comment with its own recency score.

In [ ]:
with open(DB_PATH, 'r') as f:
    db = json.load(f)

df = pd.DataFrame([{
    'recency_score': c['recency_score'],
    'comment_date': c['comment_date'],
    'comment_age_days': c['comment_age_days'],
    'comment_upvotes': c['comment_upvotes'],
    'subreddit': c['subreddit'],
    'post_title': c['post_title'],
    'comment_body': c['comment_body']
} for c in db['comments']])

# Sort by recency
df = df.sort_values('recency_score', ascending=False)

pd.set_option('display.max_colwidth', 80)
print(f'Total comments in database: {len(df)}')
df.head(90)

Total comments in database: 409


,recency_score,comment_date,comment_age_days,comment_upvotes,subreddit,post_title,comment_body
383,1.000,2026-04-18,0,1,SFFood,sf restaurant week,China Live. $45/person for dinner. Went last night and it was incredible! It...
17,1.000,2026-03-31,0,1,AskSF,Food recommendations for visiting Montana family,Hookfish by the beach is fun and delicious but Water Bar and Hog Island in t...
5,1.000,2026-04-01,0,20,AskSF,What restaurant in sf has the best spaghetti?,Pasta supply co
4,1.000,2026-04-01,0,6,AskSF,What restaurant in sf has the best spaghetti?,Firenze By Night \n\nI was gifted three day old pasta marinara. It was bette...
1,1.000,2026-04-01,0,19,AskSF,What restaurant in sf has the best spaghetti?,"Delfina’s is a classic, though I never order it because I always prefer a pr..."
...,...,...,...,...,...,...,...
49,0.967,2026-03-29,3,6,AskSF,Best DoorDash/UberEats delivery options?,"We get a few meals delivered from Cook Unity each week, it’s portioned meals..."
36,0.967,2026-03-29,3,3,AskSF,Restaurant recommendations- gift for married couple,anchovy bar is a marvelous restaurant to celebrate an anniversary.
47,0.967,2026-03-29,3,3,AskSF,Best DoorDash/UberEats delivery options?,If you have amex you get 10 bucks free on grubhub. If I’m only looking for r...
46,0.967,2026-03-29,3,6,AskSF,Best DoorDash/UberEats delivery options?,"Depends on the neighborhood.. RT Rotisserie, Keeva, Papalote"


## 7. Query the Database
Fast search against pre-cleaned comment database.
Returns comments ranked by recency score.

In [ ]:
def query_database(query, db_path=DB_PATH, top_n=15):
    """
    Search comment database for a user query.
    Searches comment body and post title for meaningful query terms.
    Returns top_n comments sorted by recency score.
    """
    with open(db_path, 'r') as f:
        db = json.load(f)

    # Extract meaningful query terms, skip generic words
    query_terms = [
        t for t in query.lower().split()
        if len(t) > 3 and t not in SKIP_WORDS
    ]

    results = []
    for comment in db['comments']:
        searchable = (comment['comment_body'] + ' ' + comment['post_title']).lower()
        if any(term in searchable for term in query_terms):
            results.append(comment)

    # Sort by comment recency
    results.sort(key=lambda x: x['recency_score'], reverse=True)
    return results[:top_n]


# Test
query = 'best ramen san francisco'
results = query_database(query)
print(f'Found {len(results)} comments for: "{query}"')

Found 15 comments for: "best ramen san francisco"


## 8. Format as LLM Context
Flat ranked list of comments — ready for the orchestration layer.

In [ ]:
def format_as_llm_context(comments, query):
    """
    Format comment rows as clean context string for the LLM.
    Each line is one community restaurant recommendation.
    Sorted by recency score so LLM sees freshest recommendations first.
    """
    context = f'Reddit community restaurant recommendations for: "{query}"\n'
    context += f'Retrieved {len(comments)} recent community comments (sorted by recency)\n'
    context += '=' * 60 + '\n\n'

    for i, comment in enumerate(comments, 1):
        context += f'RECOMMENDATION {i}\n'
        context += f'  Posted: {comment["comment_date"]} ({comment["comment_age_days"]} days ago)\n'
        context += f'  Recency Score: {comment["recency_score"]} (1.0=today, 0.0=90 days ago)\n'
        context += f'  Upvotes: {comment["comment_upvotes"]}\n'
        context += f'  Source: r/{comment["subreddit"]} | Thread: "{comment["post_title"]}"\n'
        context += f'  Comment: {comment["comment_body"][:300]}\n\n'

    return context


llm_context = format_as_llm_context(results, query)
print(llm_context[:3000])

Reddit community restaurant recommendations for: "best ramen san francisco"
Retrieved 15 recent community comments (sorted by recency)

RECOMMENDATION 1
  Posted: 2026-03-31 (0 days ago)
  Recency Score: 1.0 (1.0=today, 0.0=90 days ago)
  Upvotes: 1
  Source: r/AskSF | Thread: "Food recommendations for visiting Montana family"
  Comment: I’d recommend japantown for Ramen/Udon (Marufuku, Hinodeya, udon mugido) or Mensho on Geary. Cafe de casa has great Brazilian meals. 

Different recommendations than what you asked but probably something they can’t get in Montana.

RECOMMENDATION 2
  Posted: 2026-04-18 (0 days ago)
  Recency Score: 1.0 (1.0=today, 0.0=90 days ago)
  Upvotes: 1
  Source: r/SFFood | Thread: "Spicy Ramen Quest: Can anything in SF top Nagi, or should I just move to Palo Alto?"
  Comment: I haven’t found a great ramen in SF yet to be honest. Nagi was also a bit too rich in my opinion

RECOMMENDATION 3
  Posted: 2026-04-18 (0 days ago)
  Recency Score: 1.0 (1.0=today, 0.0=90

## 9. Database Stats

In [ ]:
with open(DB_PATH, 'r') as f:
    db = json.load(f)

df_stats = pd.DataFrame([{
    'subreddit': c['subreddit'],
    'comment_age_days': c['comment_age_days'],
    'recency_score': c['recency_score'],
    'comment_upvotes': c['comment_upvotes']
} for c in db['comments']])

print(f'Last updated: {db["last_updated"]}')
print(f'Total comments: {db["total_comments"]}')
print(f'\nComments by subreddit:')
print(df_stats['subreddit'].value_counts())
print(f'\nAge distribution:')
print(f'  0-7 days:   {len(df_stats[df_stats["comment_age_days"] <= 7])}')
print(f'  8-30 days:  {len(df_stats[(df_stats["comment_age_days"] > 7) & (df_stats["comment_age_days"] <= 30)])}')
print(f'  31-90 days: {len(df_stats[df_stats["comment_age_days"] > 30])}')
print(f'\nAvg recency score: {df_stats["recency_score"].mean():.3f}')
print(f'Avg upvotes per comment: {df_stats["comment_upvotes"].mean():.1f}')

Last updated: 2026-04-19T01:30:40.133171+00:00
Total comments: 409

Comments by subreddit:
subreddit
AskSF     257
SFFood    152
Name: count, dtype: int64

Age distribution:
  0-7 days:   180
  8-30 days:  165
  31-90 days: 64

Avg recency score: 0.810
Avg upvotes per comment: 8.9


## 10. Freeze Eval Snapshot
Fixed corpus for reproducible evaluation — **never overwrite this file**.
All retrieval experiments (Variant A and B) run against this snapshot.

In [ ]:
import shutil

EVAL_SNAPSHOT_PATH = 'eval_snapshot.json'

# Only freeze if snapshot doesn't already exist
import os
if not os.path.exists(EVAL_SNAPSHOT_PATH):
    shutil.copy(DB_PATH, EVAL_SNAPSHOT_PATH)
    print(f'Snapshot frozen → {EVAL_SNAPSHOT_PATH}')
else:
    print(f'Snapshot already exists — not overwriting ({EVAL_SNAPSHOT_PATH})')

with open(EVAL_SNAPSHOT_PATH, 'r') as f:
    snap = json.load(f)

print(f'Total comments in snapshot: {snap["total_comments"]}')
print(f'Snapshot timestamp: {snap["last_updated"]}')
print('This file is now FIXED. All eval runs against this corpus.')

Snapshot already exists — not overwriting (eval_snapshot.json)
Total comments in snapshot: 409
Snapshot timestamp: 2026-04-19T01:30:40.133171+00:00
This file is now FIXED. All eval runs against this corpus.


## 11. Generate QA Pairs
Claude Haiku reads each comment and generates the natural query a user would type to find it.
This gives us ground truth: we know which comment_id is the correct answer for each query.
Split 80/20 into validation (tuning) and test (final eval only).

In [ ]:
import random

def generate_qa_pairs(snapshot_path, n_pairs=200, seed=42):
    """
    Generate QA pairs from the frozen eval snapshot.
    Comment → Claude Haiku generates the natural query a user would type to find it.
    Ground truth: that comment_id is the relevant result for the generated query.
    """
    with open(snapshot_path, 'r') as f:
        snap = json.load(f)

    comments = snap['comments']
    random.seed(seed)

    # Filter out very short comments (< 20 chars) — not enough signal
    usable = [c for c in comments if len(c['comment_body']) >= 20]
    sampled = random.sample(usable, min(n_pairs, len(usable)))

    qa_pairs = []
    for i, comment in enumerate(sampled):
        prompt = f"""A Reddit user posted this restaurant recommendation for San Francisco:

"{comment['comment_body'][:400]}"

Write a short, natural search query (3-8 words) that someone would type to find this recommendation.
Reply with ONLY the query, no quotes or explanation."""

        try:
            response = claude.messages.create(
                model='claude-haiku-4-5-20251001',
                max_tokens=25,
                messages=[{'role': 'user', 'content': prompt}]
            )
            query = response.content[0].text.strip().strip('"').strip("'")
            qa_pairs.append({
                'query': query,
                'relevant_comment_id': comment['comment_id'],
                'relevant_comment_preview': comment['comment_body'][:150],
                'post_title': comment['post_title']
            })
            if (i + 1) % 25 == 0:
                print(f'  {i + 1}/{len(sampled)} QA pairs generated...')
        except Exception as e:
            print(f'  Error on comment {i}: {e}')

    return qa_pairs


# Load or generate QA pairs
if os.path.exists(QA_VAL_PATH) and os.path.exists(QA_TEST_PATH):
    with open(QA_VAL_PATH) as f:
        val_set = json.load(f)
    with open(QA_TEST_PATH) as f:
        test_set = json.load(f)
    print(f'Loaded existing QA sets — val: {len(val_set)}, test: {len(test_set)}')
else:
    print('Generating QA pairs from eval snapshot...')
    all_qa = generate_qa_pairs(EVAL_SNAPSHOT_PATH, n_pairs=200)

    random.seed(42)
    random.shuffle(all_qa)
    split = int(len(all_qa) * 0.8)
    val_set = all_qa[:split]
    test_set = all_qa[split:]

    with open(QA_VAL_PATH, 'w') as f:
        json.dump(val_set, f, indent=2)
    with open(QA_TEST_PATH, 'w') as f:
        json.dump(test_set, f, indent=2)

    print(f'\nGenerated {len(all_qa)} total QA pairs')
    print(f'Validation set: {len(val_set)} → {QA_VAL_PATH}')
    print(f'Test set:       {len(test_set)} → {QA_TEST_PATH}')
    print('WARNING: Do not tune hyperparameters on the test set.')

print('\nSample QA pairs:')
for qa in val_set[:3]:
    print(f'  Q: {qa["query"]}')
    print(f'  A: {qa["relevant_comment_preview"][:80]}...')
    print()

Loaded existing QA sets — val: 160, test: 40

Sample QA pairs:
  Q: bobo's san francisco restaurant recommendation
  A: Bobo’s probably doesn’t fit “not too expensive” but is otherwise perfect for thi...

  Q: Original Joes San Francisco restaurant
  A: maybe Original Joes?...

  Q: best sushi san mateo yoshizumi wakuriya
  A: out of the city but Yoshizumi for sushi and wakuriya for omakase, both in san ma...



## 12. Variant A — Keyword Search Baseline
Exact string matching on query terms against comment body + post title, ranked by recency.
This is the current production retrieval method — no understanding of synonyms or context.

In [ ]:
import numpy as np

def keyword_search_v(query, comments, top_n=10):
    """Variant A: keyword search — sparse retrieval baseline."""
    STOP = {
        'best', 'good', 'great', 'san', 'francisco', 'sf', 'restaurant',
        'place', 'spot', 'where', 'what', 'any', 'near', 'want', 'find',
        'some', 'with', 'from', 'have', 'that', 'this', 'they', 'their',
        'there', 'been', 'will', 'also', 'just', 'really', 'very'
    }
    terms = [t.lower() for t in query.split() if len(t) > 2 and t.lower() not in STOP]
    results = []
    for comment in comments:
        searchable = (comment['comment_body'] + ' ' + comment['post_title']).lower()
        if any(term in searchable for term in terms):
            results.append(comment)
    results.sort(key=lambda x: x['recency_score'], reverse=True)
    return [r['comment_id'] for r in results[:top_n]]


def evaluate_retrieval(qa_pairs, search_fn, comments, k=5):
    """
    Evaluate retrieval using Precision@k, Recall@k, MRR.
    Each QA pair has one ground-truth relevant comment_id.
    """
    precisions, recalls, rrs = [], [], []
    zero_results = 0

    for qa in qa_pairs:
        retrieved = search_fn(qa['query'], comments, top_n=k)
        relevant = qa['relevant_comment_id']

        if not retrieved:
            zero_results += 1

        hit = 1 if relevant in retrieved else 0
        precisions.append(hit / k)
        recalls.append(float(hit))
        rrs.append(1.0 / (retrieved.index(relevant) + 1) if hit else 0.0)

    return {
        f'Precision@{k}': round(np.mean(precisions), 3),
        f'Recall@{k}':    round(np.mean(recalls), 3),
        'MRR':            round(np.mean(rrs), 3),
        'Zero-result queries': zero_results
    }


# Load eval corpus and validation set
with open(EVAL_SNAPSHOT_PATH) as f:
    eval_db = json.load(f)
eval_comments = eval_db['comments']

with open(QA_VAL_PATH) as f:
    val_set = json.load(f)

print(f'Eval corpus: {len(eval_comments)} comments')
print(f'Validation queries: {len(val_set)}\n')

print('=== Variant A: Keyword Search (Baseline) ===')
results_a = evaluate_retrieval(val_set, keyword_search_v, eval_comments, k=5)
for metric, val in results_a.items():
    print(f'  {metric}: {val}')

Eval corpus: 409 comments
Validation queries: 160

=== Variant A: Keyword Search (Baseline) ===
  Precision@5: 0.057
  Recall@5: 0.288
  MRR: 0.161
  Zero-result queries: 0


## 13. Variant B — Semantic Search (Dense Retrieval)
Embeds all comments at build time using `sentence-transformers/all-MiniLM-L6-v2`.
At query time, embeds the query and ranks comments by cosine similarity.
Understands synonyms and intent — no hardcoded keywords needed.

In [ ]:
import sys
!{sys.executable} -m pip install sentence-transformers -q

import numpy as np
from sentence_transformers import SentenceTransformer

# Load embedding model (downloads once, cached after)
print('Loading sentence-transformers model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model ready.')

# Embed all comments in the eval snapshot
comment_texts = [
    c['comment_body'] + ' ' + c['post_title']
    for c in eval_comments
]
print(f'Embedding {len(comment_texts)} comments...')
comment_embeddings = embed_model.encode(comment_texts, show_progress_bar=True, batch_size=64)
print(f'Done. Embeddings shape: {comment_embeddings.shape}')

# Pre-normalize for fast cosine similarity
comment_norms = np.linalg.norm(comment_embeddings, axis=1, keepdims=True)
comment_embeddings_normed = comment_embeddings / (comment_norms + 1e-8)


def semantic_search_v(query, comments, top_n=10):
    """Variant B: dense retrieval via cosine similarity on sentence embeddings."""
    q_emb = embed_model.encode([query])
    q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-8)
    scores = (comment_embeddings_normed @ q_norm.T).flatten()
    top_idx = np.argsort(scores)[::-1][:top_n]
    return [comments[i]['comment_id'] for i in top_idx]


# Sanity check
test_q = 'cheap ramen noodle soup'
test_ids = semantic_search_v(test_q, eval_comments, top_n=3)
test_comments = {c['comment_id']: c for c in eval_comments}
print(f'\nTest query: "{test_q}"')
for cid in test_ids:
    print(f'  → {test_comments[cid]["comment_body"][:80]}')

Loading sentence-transformers model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model ready.
Embedding 409 comments...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Done. Embeddings shape: (409, 384)

Test query: "cheap ramen noodle soup"
  → Ok hear me out. 

Trust me with this. 

Instead of ramen, get the braised beef n
  → Not exactly ramen, but Daeho beef soup in Japantown will definitely scratch that
  → Ramenwelll Souper Spicy Garlic Ramen baaaaayyyybyyyyy! It’s so damn good! 


## 14. Results — Variant A vs Variant B
Evaluate both methods on the same validation set and same corpus.
**Do not run on the test set until final report submission.**

In [ ]:
K = 5  # top-k for all metrics

print('=== Variant A: Keyword Search (Sparse Retrieval) ===')
results_a = evaluate_retrieval(val_set, keyword_search_v, eval_comments, k=K)
for metric, val in results_a.items():
    print(f'  {metric}: {val}')

print('\n=== Variant B: Semantic Search (Dense Retrieval) ===')
results_b = evaluate_retrieval(val_set, semantic_search_v, eval_comments, k=K)
for metric, val in results_b.items():
    print(f'  {metric}: {val}')

# Comparison table
print('\n' + '=' * 55)
print(f'{"Metric":<20} {"Variant A (Keyword)":<20} {"Variant B (Semantic)"}')
print('=' * 55)
for metric in [f'Precision@{K}', f'Recall@{K}', 'MRR']:
    a = results_a[metric]
    b = results_b[metric]
    delta = b - a
    sign = '+' if delta >= 0 else ''
    winner = '← B wins' if delta > 0 else ('← A wins' if delta < 0 else 'tie')
    print(f'{metric:<20} {a:<20} {b}  ({sign}{delta:.3f}) {winner}')
print('=' * 55)
print(f'{"Zero-result queries":<20} {results_a["Zero-result queries"]:<20} {results_b["Zero-result queries"]}')

# Save results for report
comparison = {
    'corpus': EVAL_SNAPSHOT_PATH,
    'eval_set': QA_VAL_PATH,
    'n_queries': len(val_set),
    'k': K,
    'variant_a_keyword': results_a,
    'variant_b_semantic': results_b
}
with open('eval_results_validation.json', 'w') as f:
    json.dump(comparison, f, indent=2)
print('\nResults saved → eval_results_validation.json')

=== Variant A: Keyword Search (Sparse Retrieval) ===
  Precision@5: 0.057
  Recall@5: 0.288
  MRR: 0.161
  Zero-result queries: 0

=== Variant B: Semantic Search (Dense Retrieval) ===
  Precision@5: 0.144
  Recall@5: 0.719
  MRR: 0.674
  Zero-result queries: 0

Metric               Variant A (Keyword)  Variant B (Semantic)
Precision@5          0.057                0.144  (+0.087) ← B wins
Recall@5             0.288                0.719  (+0.431) ← B wins
MRR                  0.161                0.674  (+0.513) ← B wins
Zero-result queries  0                    0

Results saved → eval_results_validation.json


---
## 15. V2 Database Build
**Improvements over V1:**
- 3 subreddits (added r/sanfrancisco)
- Removed hardcoded keyword filter — Haiku classifies all posts directly
- TOP_COMMENTS increased from 10 → 25
- Saves to `master_database_v2.json` (team handoff file)

In [ ]:
# ============================================================
# 15. V2 Database Build (patched)
# ============================================================
# Fixes from previous V2 run:
#   1. Classifier prompt broadened — now includes ice cream shops, cafes, bars,
#      bakeries, dessert spots (previously excluded by strict "restaurant" wording)
#   2. Retry logic on classifier — silent 500-error rejections were killing AskSF
#   3. Dropped 'top year' feed — redundant with other feeds, doubled scrape time
#   4. Fetch comments once per post, reuse for classifier + extraction
#   5. Age-filter before any comment fetching

import time
from datetime import datetime, timezone

# V2 Configuration
SUBREDDITS_V2 = ['AskSF', 'SFFood']
TOP_COMMENTS_V2 = 25
POST_LIMIT_V2 = 100
MAX_AGE_DAYS_V2 = 365
DB_PATH_V2 = 'master_database_v2.json'


def llm_classifier_v2(post_title, post_body, comments_preview, max_retries=3):
    """
    Broader food/drink classifier. Retries on API errors instead of
    silently treating transient failures as rejections.
    """
    prompt = f"""Is this Reddit post asking for or giving recommendations for any food or drink establishment in San Francisco? This includes restaurants, cafes, bars, ice cream shops, bakeries, dessert spots, food trucks, coffee shops, and similar places.

Title: {post_title}
Body: {post_body[:200]}
Top comments preview: {comments_preview[:300]}

Answer yes or no only."""
    for attempt in range(max_retries):
        try:
            response = claude.messages.create(
                model='claude-haiku-4-5-20251001',
                max_tokens=5,
                messages=[{'role': 'user', 'content': prompt}]
            )
            return response.content[0].text.strip().lower() == 'yes'
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # 1s, 2s, 4s backoff
                continue
            print(f'Classifier failed after {max_retries} retries: {e}')
            return False
    return False


def build_v2_database(db_path=DB_PATH_V2):
    """
    V2 build — no keyword filter, 2 subreddits, 25 comments per post, 1-year window.
    Broader classifier + retry logic + single comment fetch per post.
    """
    try:
        with open(db_path, 'r') as f:
            existing = json.load(f)
        existing_comments = {c['comment_id']: c for c in existing['comments']}
        seen_post_ids = set(existing.get('processed_post_ids', []))
        print(f'Loaded existing V2 database: {len(existing_comments)} comments')
    except FileNotFoundError:
        existing_comments = {}
        seen_post_ids = set()
        print('No existing V2 database — building fresh')

    new_comments = 0
    rejected_llm = 0
    new_posts_processed = 0
    start_time = time.time()

    for sub_name in SUBREDDITS_V2:
        print(f'\nScraping r/{sub_name}...')
        subreddit = reddit.subreddit(sub_name)

        # Dropped top(year) — it largely duplicates top(month) + hot + new
        feeds = [
            subreddit.new(limit=POST_LIMIT_V2),
            subreddit.hot(limit=POST_LIMIT_V2),
            subreddit.top(limit=POST_LIMIT_V2, time_filter='month'),
        ]

        seen_in_this_run = set()

        for feed in feeds:
            for post in feed:
                if post.id in seen_post_ids or post.id in seen_in_this_run:
                    continue
                seen_in_this_run.add(post.id)

                # Age-filter BEFORE any network calls on the post's comments
                if get_age_days(post.created_utc) > MAX_AGE_DAYS_V2:
                    continue

                # Fetch comments once, reuse for both classifier and extraction
                try:
                    post.comments.replace_more(limit=0)
                except Exception as e:
                    print(f'  Skipping post {post.id} (comment fetch failed: {e})')
                    continue

                comments_preview = ' '.join([
                    c.body[:100] for c in post.comments[:3]
                    if c.body not in ['[removed]', '[deleted]']
                ])

                if not llm_classifier_v2(post.title, post.selftext, comments_preview):
                    rejected_llm += 1
                    seen_post_ids.add(post.id)
                    continue

                # Extract up to 25 comments (reuse the already-fetched comments)
                for comment in post.comments[:TOP_COMMENTS_V2]:
                    age_days = get_age_days(comment.created_utc)
                    if age_days > MAX_AGE_DAYS_V2:
                        continue
                    if not comment.body or comment.body in ['[removed]', '[deleted]']:
                        continue
                    if comment.id not in existing_comments:
                        existing_comments[comment.id] = {
                            'comment_id': comment.id,
                            'comment_body': comment.body,
                            'comment_date': format_timestamp(comment.created_utc),
                            'comment_age_days': age_days,
                            'recency_score': compute_recency_score(age_days),
                            'comment_upvotes': comment.score,
                            'post_title': post.title,
                            'post_url': f'https://reddit.com{post.permalink}',
                            'subreddit': sub_name
                        }
                        new_comments += 1

                seen_post_ids.add(post.id)
                new_posts_processed += 1

                # Light progress indicator every 25 posts
                if new_posts_processed % 25 == 0:
                    elapsed = time.time() - start_time
                    print(f'  Processed {new_posts_processed} posts '
                          f'({new_comments} comments, {elapsed:.0f}s elapsed)')

    # Remove expired comments
    existing_comments = {
        cid: c for cid, c in existing_comments.items()
        if c['comment_age_days'] <= MAX_AGE_DAYS_V2
    }

    output = {
        'version': 'v2',
        'last_updated': datetime.now(tz=timezone.utc).isoformat(),
        'total_comments': len(existing_comments),
        'subreddits': SUBREDDITS_V2,
        'max_age_days': MAX_AGE_DAYS_V2,
        'top_comments_per_post': TOP_COMMENTS_V2,
        'processed_post_ids': list(seen_post_ids),
        'comments': list(existing_comments.values())
    }

    with open(db_path, 'w') as f:
        json.dump(output, f, indent=2)

    total_elapsed = time.time() - start_time
    print(f'\n=== V2 Database Build Complete ===')
    print(f'Runtime: {total_elapsed/60:.1f} minutes')
    print(f'Subreddits: {SUBREDDITS_V2}')
    print(f'New posts processed: {new_posts_processed}')
    print(f'New comments added: {new_comments}')
    print(f'Rejected by Haiku classifier: {rejected_llm}')
    print(f'Total comments in V2 database: {len(existing_comments)}')
    print(f'Saved → {db_path}')

    return list(existing_comments.values())


# Delete existing V2 to rebuild from scratch with broader classifier
import os
if os.path.exists(DB_PATH_V2):
    os.remove(DB_PATH_V2)
    print(f'Deleted existing {DB_PATH_V2} — rebuilding from scratch')

comments_v2 = build_v2_database()
print(f'\nV2 ready — {len(comments_v2)} comments')


# ---------- Sanity check ----------
ice_cream_count = sum(
    1 for c in comments_v2
    if 'ice cream' in c['comment_body'].lower() or 'ice cream' in c['post_title'].lower()
)
from collections import Counter
sub_dist = Counter(c['subreddit'] for c in comments_v2)
print(f'\nSanity checks:')
print(f'  Ice cream mentions: {ice_cream_count} (should be >20 now)')
print(f'  Subreddit distribution: {dict(sub_dist)}')

In [ ]:
with open(DB_PATH_V2, 'r') as f:
    db_v2 = json.load(f)

df_v2 = pd.DataFrame([{
    'recency_score': c['recency_score'],
    'comment_date': c['comment_date'],
    'comment_age_days': c['comment_age_days'],
    'comment_upvotes': c['comment_upvotes'],
    'subreddit': c['subreddit'],
    'post_title': c['post_title'],
    'comment_body': c['comment_body']
} for c in db_v2['comments']])

df_v2 = df_v2.sort_values('recency_score', ascending=False)

print(f'Version: {db_v2["version"]}')
print(f'Last updated: {db_v2["last_updated"]}')
print(f'Total comments: {db_v2["total_comments"]}')
print(f'Subreddits: {db_v2["subreddits"]}')
print(f'\nComments by subreddit:')
print(df_v2['subreddit'].value_counts())
print(f'\nAge distribution:')
print(f'  0-7 days:   {len(df_v2[df_v2["comment_age_days"] <= 7])}')
print(f'  8-30 days:  {len(df_v2[(df_v2["comment_age_days"] > 7) & (df_v2["comment_age_days"] <= 30)])}')
print(f'  31-90 days: {len(df_v2[df_v2["comment_age_days"] > 30])}')
print(f'\nAvg recency score: {df_v2["recency_score"].mean():.3f}')
print(f'Avg upvotes per comment: {df_v2["comment_upvotes"].mean():.1f}')
print()
df_v2.head(20)

Version: v2
Last updated: 2026-04-20T05:03:16.368480+00:00
Total comments: 1478
Subreddits: ['AskSF', 'SFFood']

Comments by subreddit:
subreddit
AskSF     842
SFFood    636
Name: count, dtype: int64

Age distribution:
  0-7 days:   458
  8-30 days:  661
  31-90 days: 359

Avg recency score: 0.768
Avg upvotes per comment: 7.3



,recency_score,comment_date,comment_age_days,comment_upvotes,subreddit,post_title,comment_body
26,1.0,2026-04-19,0,2,AskSF,Steam or California Common style beer on tap in San Francisco?,San Francisco Brewing Co in Ghirardelli Square usually has a California Comm...
0,1.0,2026-04-20,0,1,AskSF,Olive loaf,[rize up](https://rizeupsourdough.com)
15,1.0,2026-04-19,0,5,AskSF,What’s the move for 4/20?,Rain or shine the stoners will show up.
14,1.0,2026-04-20,0,1,AskSF,Roses pizzeria,"Last I heard, they were targeting a March opening so hopefully whatever is c..."
13,1.0,2026-04-19,0,2,AskSF,Roses pizzeria,I think they’re still hiring given the few job ads I’ve seen for a new pizze...
12,1.0,2026-04-19,0,2,AskSF,Roses pizzeria,Been walking past there couple times this week but haven't seen much activit...
11,1.0,2026-04-20,0,1,AskSF,Spicy ramen - similar to a place I found in Japan.,Insta: https://www.instagram.com/kikanbo_japan?igsh=MzRlODBiNWFlZA==
10,1.0,2026-04-20,0,1,AskSF,Spicy ramen - similar to a place I found in Japan.,Linke: https://kikanbo.co.jp/
9,1.0,2026-04-20,0,2,AskSF,Spicy ramen - similar to a place I found in Japan.,This was asked last week: see if anyone has suggestions here\n\n[https://www...
8,1.0,2026-04-20,0,4,AskSF,Spicy ramen - similar to a place I found in Japan.,"I’ve been to Kikanbo many times, it’s a gem. Unfortunately no place in San F..."


In [ ]:
# ============================================================
# V2 Semantic Search — loads V2 corpus and builds embeddings
# ============================================================

import numpy as np

# Load V2 comments from Drive
with open(DB_PATH_V2, 'r') as f:
    db_v2 = json.load(f)
eval_comments_v2 = db_v2['comments']

print(f'Loaded {len(eval_comments_v2)} V2 comments from {DB_PATH_V2}')

# Embed V2 comments
comment_texts_v2 = [
    c['comment_body'] + ' ' + c['post_title']
    for c in eval_comments_v2
]
print(f'Embedding {len(comment_texts_v2)} V2 comments...')
comment_embeddings_v2 = embed_model.encode(
    comment_texts_v2, show_progress_bar=False, batch_size=64
)
comment_norms_v2 = np.linalg.norm(comment_embeddings_v2, axis=1, keepdims=True)
comment_embeddings_normed_v2 = comment_embeddings_v2 / (comment_norms_v2 + 1e-8)
print(f'Done. Embeddings shape: {comment_embeddings_normed_v2.shape}')


# ============================================================
# Hybrid Semantic Search — Phrase-Match Boosted
# ============================================================

def semantic_search_v2(query, top_n=15):
    """
    Hybrid retrieval: semantic similarity boosted by exact phrase match.
    Fixes false positives where 'ice' matches 'service' or 'cream' matches 'creamy'.
    """
    q_emb = embed_model.encode([query])
    q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-8)
    scores = (comment_embeddings_normed_v2 @ q_norm.T).flatten().copy()

    # Clean the query into a phrase for matching
    q_clean = query.lower()
    for stop in ['in san francisco', 'san francisco', ' sf ', 'best ', 'good ',
                 'great ', 'top ', 'shop', 'shops', 'place', 'places']:
        q_clean = q_clean.replace(stop, ' ')
    q_clean = ' '.join(q_clean.split())

    match_phrases = [q_clean] if len(q_clean) > 2 else []
    individual_words = [w for w in q_clean.split() if len(w) > 3]

    if match_phrases or individual_words:
        for i, c in enumerate(eval_comments_v2):
            searchable = (c['comment_body'] + ' ' + c['post_title']).lower()
            if any(p in searchable for p in match_phrases):
                scores[i] *= 2.0  # strong boost for full phrase
            elif any(w in searchable.split() for w in individual_words):
                scores[i] *= 1.3  # weaker boost for individual word

    top_idx = np.argsort(scores)[::-1][:top_n]
    return [eval_comments_v2[i] for i in top_idx]


print('V2 semantic search ready with phrase-match boost')

Loaded 1478 V2 comments from /content/drive/MyDrive/belly button/master_database_v2.json
Embedding 1478 V2 comments...
Done. Embeddings shape: (1478, 384)
V2 semantic search ready with phrase-match boost


In [ ]:
# ============================================================
# 16. Reddit Recommendation Pipeline — Team Comparison Output
# ============================================================
# Runs end-to-end on V2:
#   query -> semantic_search_v2 -> LLM extract restaurants -> aggregate -> rank -> format
#
# Reuses: eval_comments_v2, comment_embeddings_normed_v2, embed_model, semantic_search_v2
# from the V2 semantic search cell above.

import json
import math
import re
from collections import defaultdict

# Pipeline Configuration
TOP_COMMENTS_RETRIEVE = 10
TOP_RESTAURANTS_OUTPUT = 5
MAX_COMMENT_CHARS_FOR_LLM = 500
SAMPLE_COMMENT_CHARS = 250
EMBEDDING_MODEL_V2 = 'sentence-transformers/all-MiniLM-L6-v2'

# Scoring weights — INITIAL HEURISTICS, tune on validation set for final report
W_RECENCY = 1.0
W_UPVOTES = 0.3
W_DIVERSITY = 0.5


def extract_restaurants_from_comment(comment_body):
    """
    Haiku reads a comment and returns a JSON list of restaurant names.
    PRODUCTION NOTE: move to build time to eliminate per-query LLM cost.
    """
    prompt = f"""Extract all restaurant, cafe, bar, or food establishment names mentioned in this Reddit comment.
Return ONLY a JSON array of names — no commentary, no explanation.
If no specific places are named, return [].

Comment: {comment_body[:MAX_COMMENT_CHARS_FOR_LLM]}

Examples:
"Try Nopalito and Tartine" -> ["Nopalito", "Tartine"]
"I love the burritos there" -> []
"Bi-Rite is overrated, Garden Creamery is better" -> ["Bi-Rite", "Garden Creamery"]
"""
    try:
        response = claude.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=150,
            messages=[{'role': 'user', 'content': prompt}]
        )
        text = response.content[0].text.strip()
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.MULTILINE).strip()
        names = json.loads(text)
        if isinstance(names, list):
            return [str(n).strip() for n in names if n and isinstance(n, (str, int, float))]
        return []
    except Exception:
        return []


def normalize_restaurant_key(name):
    """Dedup key so 'Bi-Rite Creamery' / 'Bi-Rite' / 'bi rite' collapse to one."""
    key = name.lower().strip()
    key = re.sub(r"[\-'\s\.,&]", '', key)
    for suffix in ['restaurant', 'cafe', 'bar', 'creamery', 'bakery', 'sf', 'thesf']:
        if key.endswith(suffix) and len(key) > len(suffix) + 2:
            key = key[:-len(suffix)]
    return key or name.lower()


def aggregate_and_rank(comments, top_k=TOP_RESTAURANTS_OUTPUT):
    restaurants = defaultdict(lambda: {'display_name': None, 'mentions': []})

    print(f'Extracting restaurants from {len(comments)} comments...')
    for comment in comments:
        names = extract_restaurants_from_comment(comment['comment_body'])
        for name in names:
            key = normalize_restaurant_key(name)
            if not key:
                continue
            current = restaurants[key]['display_name']
            if current is None or len(name) > len(current):
                restaurants[key]['display_name'] = name
            restaurants[key]['mentions'].append({
                'comment_id': comment['comment_id'],
                'body': comment['comment_body'],
                'recency_score': comment['recency_score'],
                'upvotes': comment.get('comment_upvotes', 0),
                'subreddit': comment['subreddit'],
                'date': comment['comment_date'],
                'age_days': comment['comment_age_days'],
                'post_title': comment['post_title'],
                'url': comment['post_url']
            })

    ranked = []
    for key, data in restaurants.items():
        mentions = data['mentions']
        recency_sum = sum(m['recency_score'] for m in mentions)
        upvote_boost = sum(math.log1p(max(m['upvotes'], 0)) for m in mentions)
        subreddit_diversity = len(set(m['subreddit'] for m in mentions))
        score = (W_RECENCY * recency_sum
                 + W_UPVOTES * upvote_boost
                 + W_DIVERSITY * subreddit_diversity)
        ranked.append({
            'display_name': data['display_name'],
            'key': key,
            'mentions': mentions,
            'mention_count': len(mentions),
            'score': round(score, 3),
            'avg_recency': round(recency_sum / len(mentions), 3),
            'max_upvotes': max((m['upvotes'] for m in mentions), default=0),
            'subreddits': sorted(set(m['subreddit'] for m in mentions)),
            'latest_date': max(m['date'] for m in mentions),
        })

    ranked.sort(key=lambda x: x['score'], reverse=True)
    return ranked[:top_k]


def format_reddit_result(query, ranked, n_retrieved):
    """Format aggregated Reddit results for the orchestration layer (team-aligned)."""
    subs_display = ' + '.join(f'r/{s}' for s in SUBREDDITS_V2)

    lines = []
    lines.append(f'Reddit restaurant recommendations for: "{query}"')
    lines.append(f'Retrieved {len(ranked)} restaurants (sorted by community score)')
    lines.append(f'Retrieval: semantic search ({EMBEDDING_MODEL_V2}) over {n_retrieved} top-matched comments')
    lines.append(f'Corpus: {subs_display}, {MAX_AGE_DAYS_V2}-day window, up to {TOP_COMMENTS_V2} comments per post')
    lines.append(f'Scoring: {W_RECENCY}×recency_sum + {W_UPVOTES}×log_upvotes + {W_DIVERSITY}×subreddit_diversity')
    lines.append('=' * 60)
    lines.append('')

    if not ranked:
        lines.append('No restaurants extracted from the retrieved comments.')
        lines.append('(Query may be under-represented in the Reddit corpus. Orchestrator should rely on Yelp/Google for this category.)')
        return '\n'.join(lines)

    for i, r in enumerate(ranked, 1):
        top_mention = max(r['mentions'], key=lambda m: (m['upvotes'], m['recency_score']))
        sample = top_mention['body'][:SAMPLE_COMMENT_CHARS]
        if len(top_mention['body']) > SAMPLE_COMMENT_CHARS:
            sample += '...'

        subs_str = ', '.join(f'r/{s}' for s in r['subreddits'])
        unique_comments = len(set(m['comment_id'] for m in r['mentions']))

        lines.append(f'RESTAURANT {i}')
        lines.append(f"  Name: {r['display_name']}")
        lines.append(f"  Community Score: {r['score']} (higher = more buzz)")
        lines.append(f"  Mention Count: {r['mention_count']} (across {unique_comments} comments)")
        lines.append(f"  Avg Recency Score: {r['avg_recency']} (1.0=today, 0.0={MAX_AGE_DAYS_V2} days ago)")
        lines.append(f"  Top Upvoted Mention: {r['max_upvotes']} upvotes")
        lines.append(f"  Subreddits: {subs_str}")
        lines.append(f"  Latest Mention: {r['latest_date']}")
        lines.append(f"  Community Quote:")
        lines.append(f'    "{sample}"')
        lines.append(f"    — r/{top_mention['subreddit']}, {top_mention['date']}, {top_mention['upvotes']} upvotes")
        lines.append(f"  Source: Reddit | {top_mention['url']}")
        lines.append('')

    return '\n'.join(lines)


def run_reddit_pipeline(query, verbose=True):
    """End-to-end: query in, formatted team-shareable output out. Uses V2 semantic search."""
    comments = semantic_search_v2(query, top_n=TOP_COMMENTS_RETRIEVE)
    ranked = aggregate_and_rank(comments, top_k=TOP_RESTAURANTS_OUTPUT)
    output = format_reddit_result(query, ranked, len(comments))
    if verbose:
        print(output)
    return output, ranked


# Run the team's test query
test_queries = [
    'Taco Shop in San Francisco'
]

all_results = {}
for q in test_queries:
    print(f"\n{'='*70}")
    output, ranked = run_reddit_pipeline(q)
    all_results[q] = output

with open('reddit_results_team_comparison.txt', 'w') as f:
    f.write('\n\n'.join(all_results.values()))

print('\n' + '='*70)
print('Saved to reddit_results_team_comparison.txt — ready to share with the team.')


Extracting restaurants from 10 comments...
Reddit restaurant recommendations for: "Taco Shop in San Francisco"
Retrieved 5 restaurants (sorted by community score)
Retrieval: semantic search (sentence-transformers/all-MiniLM-L6-v2) over 10 top-matched comments
Corpus: r/AskSF + r/SFFood, 365-day window, up to 25 comments per post
Scoring: 1.0×recency_sum + 0.3×log_upvotes + 0.5×subreddit_diversity

RESTAURANT 1
  Name: Taco Bell Cantina
  Community Score: 5.1 (higher = more buzz)
  Mention Count: 3 (across 3 comments)
  Avg Recency Score: 0.878 (1.0=today, 0.0=365 days ago)
  Top Upvoted Mention: 116 upvotes
  Subreddits: r/AskSF
  Latest Mention: 2026-04-11
  Community Quote:
    "Taco Bell cantina "
    — r/AskSF, 2026-04-06, 116 upvotes
  Source: Reddit | https://reddit.com/r/AskSF/comments/1sdr1b5/looking_for_the_most_authentic_restaurants_in_sf/

RESTAURANT 2
  Name: Tacos el patron
  Community Score: 3.667 (higher = more buzz)
  Mention Count: 3 (across 3 comments)
  Avg Recency 